# 第143章: DreamerV3 — 夢の中で数千回試行する世界モデル

## 📋 この章で学ぶこと

この章を終えると、以下ができるようになります：

- [ ] RSSM（確定的+確率的状態遷移）の仕組みを説明できる
- [ ] 世界モデルの3フェーズ学習（世界モデル→想像→データ収集）を実装できる
- [ ] Imagination Rolloutで夢の中の軌跡を生成できる
- [ ] Actor-Criticを潜在空間で訓練できる
- [ ] DreamerV3のアーキテクチャと既存手法との違いを説明できる

## 🎯 前提知識

- ✅ Notebook 142（モデルベース強化学習 / Dyna-Q）
- ✅ Notebook 36（VAE / PyTorch基礎）
- ✅ Notebook 141（JEPA / 表現学習）

⏱️ **推定学習時間**: 180-240分
📊 **難易度**: ★★★★★（実践）
🎓 **カテゴリ**: 世界モデル

## 目次

1. [DreamerV3の概要と動機](#section1)
2. [簡易環境の構築](#section2)
3. [RSSM — Recurrent State Space Model](#section3)
4. [世界モデルの構築と訓練](#section4)
5. [Actor-Critic in 潜在空間](#section5)
6. [Imagination Rollout](#section6)
7. [SimpleDreamer 統合パイプライン](#section7)
8. [夢の可視化](#section8)
9. [まとめ・よくあるエラー・自己評価クイズ](#summary)

In [ ]:
# ============================================================
# 環境設定
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import warnings

warnings.filterwarnings('ignore')

# 日本語フォント設定
def setup_japanese_font():
    japanese_fonts = [
        'Hiragino Sans', 'Hiragino Maru Gothic Pro',
        'Yu Gothic', 'MS Gothic',
        'Noto Sans CJK JP', 'IPAexGothic',
    ]
    available_fonts = set(f.name for f in fm.fontManager.ttflist)
    for font in japanese_fonts:
        if font in available_fonts:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return font
    return None

font_used = setup_japanese_font()
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'✅ ライブラリのインポート完了')
print(f'🖥️ デバイス: {device}')
print(f'📝 日本語フォント: {font_used}')

<a id="section1"></a>
## 1. DreamerV3の概要と動機

### 🤔 なぜDreamerV3が必要か？

Notebook 142でDyna-Qを学びました。Dyna-Qはテーブルベースの世界モデルで「想像」しましたが、以下の限界がありました：

| 限界 | Dyna-Q | DreamerV3 |
|------|--------|-----------|
| 状態表現 | 離散テーブル | ニューラルネット潜在空間 |
| 遷移モデル | テーブル記録 | RSSM (GRU + 確率モデル) |
| 想像の長さ | 1ステップ | 多ステップロールアウト |
| 方策学習 | Q値更新 | Actor-Critic（勾配ベース） |
| 画像観測 | 不可 | CNN エンコーダ/デコーダ |

### 📊 DreamerV3の3つのフェーズ

```
フェーズ1: 世界モデル学習
  実データ (o, a, r) → エンコーダ + RSSM + デコーダ + 報酬予測を訓練

フェーズ2: 想像の中で方策学習
  RSSM内でロールアウト → Actor-Criticを勾配で更新（実環境不要！）

フェーズ3: 実環境でデータ収集
  学習した方策で行動 → 新しいデータをバッファに追加
```

このサイクルを繰り返すことで、少ない実環境インタラクションで効率的に学習します。

In [ ]:
# ============================================================
# DreamerV3 アーキテクチャ概念図
# ============================================================

def visualize_dreamer_architecture():
    fig, ax = plt.subplots(figsize=(16, 8))
    
    # フェーズ1: 世界モデル
    boxes = [
        (0.05, 0.7, 0.12, 0.15, '観測\no_t', 'lightcoral'),
        (0.20, 0.7, 0.12, 0.15, 'エンコーダ\nCNN', 'lightskyblue'),
        (0.38, 0.7, 0.16, 0.15, 'RSSM\nh_t, z_t', 'lightgreen'),
        (0.60, 0.7, 0.12, 0.15, 'デコーダ\nCNN^T', 'lightskyblue'),
        (0.60, 0.45, 0.12, 0.15, '報酬予測\nMLP', 'wheat'),
        (0.78, 0.7, 0.12, 0.15, '再構成\nô_t', 'lightcoral'),
    ]
    
    for x, y, w, h, label, color in boxes:
        rect = plt.Rectangle((x, y), w, h, facecolor=color, 
                             edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center', 
               fontsize=9, fontweight='bold')
    
    # 矢印
    arrows = [(0.17, 0.775, 0.03, 0), (0.32, 0.775, 0.06, 0),
              (0.54, 0.775, 0.06, 0), (0.72, 0.775, 0.06, 0),
              (0.46, 0.70, 0, -0.10)]
    for x, y, dx, dy in arrows:
        ax.annotate('', xy=(x+dx, y+dy), xytext=(x, y),
                    arrowprops=dict(arrowstyle='->', lw=2))
    
    # フェーズ2: Actor-Critic
    boxes2 = [
        (0.05, 0.15, 0.16, 0.15, 'RSSM\n(想像)', 'palegreen'),
        (0.28, 0.22, 0.12, 0.12, 'Actor\nπ(a|s)', 'plum'),
        (0.28, 0.05, 0.12, 0.12, 'Critic\nV(s)', 'lightyellow'),
        (0.48, 0.15, 0.14, 0.15, '想像\nロールアウト', 'lightsalmon'),
    ]
    
    for x, y, w, h, label, color in boxes2:
        rect = plt.Rectangle((x, y), w, h, facecolor=color,
                             edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + w/2, y + h/2, label, ha='center', va='center',
               fontsize=9, fontweight='bold')
    
    # フェーズラベル
    ax.text(0.45, 0.92, 'フェーズ1: 世界モデル学習（実データ使用）', 
           fontsize=13, fontweight='bold', ha='center',
           bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    ax.text(0.35, 0.38, 'フェーズ2: 想像の中で方策学習（実環境不要）',
           fontsize=13, fontweight='bold', ha='center',
           bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title('DreamerV3 アーキテクチャ概要', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_dreamer_architecture()

<a id="section2"></a>
## 2. 簡易環境の構築

DreamerV3の各コンポーネントを理解するため、シンプルな1D環境を作ります。

**ボールキャッチ環境:**
- 状態: ボールの位置 x ∈ [-1, 1] と速度 v
- 行動: 力 a ∈ [-1, 1]（連続値）
- 目標: ボールを原点 x=0 に保つ
- 報酬: -|x|（原点に近いほど高い報酬）
- 観測: 状態に微小ノイズを加えた5次元ベクトル（冗長な次元を含む）

In [ ]:
# ============================================================
# シンプルな連続制御環境
# ============================================================

class BallBalanceEnv:
    # ボールを原点に保つ1D制御タスク
    
    def __init__(self):
        self.dt = 0.1          # タイムステップ
        self.obs_dim = 5       # 観測次元（冗長次元含む）
        self.action_dim = 1    # 行動次元
        self.max_steps = 50    # エピソード長
        self.reset()
    
    def reset(self):
        self.x = np.random.uniform(-0.5, 0.5)  # 位置
        self.v = np.random.uniform(-0.2, 0.2)  # 速度
        self.step_count = 0
        return self._get_obs()
    
    def _get_obs(self):
        # 位置と速度に加え、冗長な特徴を含む5次元観測
        noise = np.random.normal(0, 0.01, 3)
        return np.array([self.x, self.v, self.x**2, 
                        np.sin(self.x), self.v * self.x]) +                np.concatenate([[0, 0], noise])
    
    def step(self, action):
        a = np.clip(action, -1, 1)
        # 物理シミュレーション
        self.v += a * self.dt
        self.v *= 0.95  # 摩擦
        self.x += self.v * self.dt
        self.x = np.clip(self.x, -1, 1)
        self.step_count += 1
        
        reward = -abs(self.x)  # 原点に近いほど高い報酬
        done = self.step_count >= self.max_steps
        return self._get_obs(), reward, done

# テスト
env = BallBalanceEnv()
obs = env.reset()
print(f"環境の初期観測: {obs}")
print(f"観測次元: {env.obs_dim}, 行動次元: {env.action_dim}")

# ランダム方策でエピソード実行
total_reward = 0
for _ in range(50):
    action = np.random.uniform(-1, 1)
    obs, r, done = env.step(action)
    total_reward += r
print(f"ランダム方策の累積報酬: {total_reward:.2f}")

In [ ]:
# ============================================================
# データ収集（ランダム方策）
# ============================================================

def collect_random_data(env, n_episodes=200):
    # ランダム方策でデータを収集
    episodes = []
    for ep in range(n_episodes):
        obs_list, act_list, rew_list = [], [], []
        obs = env.reset()
        done = False
        while not done:
            action = np.random.uniform(-1, 1)
            obs_list.append(obs)
            act_list.append(action)
            next_obs, reward, done = env.step(action)
            rew_list.append(reward)
        episodes.append({
            'observations': np.array(obs_list),
            'actions': np.array(act_list),
            'rewards': np.array(rew_list),
        })
    return episodes

env = BallBalanceEnv()
data = collect_random_data(env, n_episodes=200)
print(f"収集エピソード数: {len(data)}")
print(f"エピソード長: {len(data[0]['observations'])}")
print(f"観測形状: {data[0]['observations'].shape}")
print(f"平均報酬: {np.mean([np.sum(ep['rewards']) for ep in data]):.2f}")

<a id="section3"></a>
## 3. RSSM — Recurrent State Space Model

### 📊 RSSMの核心アイデア

RSSMは**確定的パス**と**確率的パス**を組み合わせた状態遷移モデルです。

```
状態 s_t = (h_t, z_t)
  h_t: 確定的状態（GRUの隠れ状態） — 長期記憶を保持
  z_t: 確率的状態（ガウシアン潜在変数） — 不確実性を表現
```

**事後分布（Posterior）**: 観測を見た後の状態推定
$$q(z_t | h_t, o_t) = \mathcal{N}(\mu_{post}, \sigma_{post})$$

**事前分布（Prior）**: 観測なしの状態予測
$$p(z_t | h_t) = \mathcal{N}(\mu_{prior}, \sigma_{prior})$$

事後と事前のKLダイバージェンスを最小化することで、観測なしでも正確な予測ができるようになります。

In [ ]:
# ============================================================
# RSSM (Recurrent State Space Model) の実装
# ============================================================

class RSSM(nn.Module):
    # 確定的+確率的状態遷移モデル
    
    def __init__(self, obs_dim=5, action_dim=1, det_dim=32, stoch_dim=16):
        super().__init__()
        self.det_dim = det_dim      # 確定的状態の次元
        self.stoch_dim = stoch_dim  # 確率的状態の次元
        
        # GRU: 確定的状態の更新
        self.gru = nn.GRUCell(stoch_dim + action_dim, det_dim)
        
        # 事前分布: h_t → (mu_prior, sigma_prior)
        self.prior_net = nn.Sequential(
            nn.Linear(det_dim, 64), nn.ELU(),
            nn.Linear(64, stoch_dim * 2)  # mu + log_sigma
        )
        
        # 事後分布: (h_t, obs_embed) → (mu_post, sigma_post)
        self.posterior_net = nn.Sequential(
            nn.Linear(det_dim + obs_dim, 64), nn.ELU(),
            nn.Linear(64, stoch_dim * 2)  # mu + log_sigma
        )
    
    def initial_state(self, batch_size):
        # 初期状態 (h_0, z_0) をゼロで初期化
        h = torch.zeros(batch_size, self.det_dim)
        z = torch.zeros(batch_size, self.stoch_dim)
        return h, z
    
    def observe_step(self, h_prev, z_prev, action, obs_embed):
        # 1ステップの状態更新（観測あり = 事後分布を使用）
        # GRU入力: 前の確率的状態 + 行動
        gru_input = torch.cat([z_prev, action], dim=-1)
        h = self.gru(gru_input, h_prev)
        
        # 事前分布
        prior_params = self.prior_net(h)
        prior_mu, prior_log_sigma = prior_params.chunk(2, dim=-1)
        prior_sigma = F.softplus(prior_log_sigma) + 0.1
        
        # 事後分布（観測を使用）
        post_input = torch.cat([h, obs_embed], dim=-1)
        post_params = self.posterior_net(post_input)
        post_mu, post_log_sigma = post_params.chunk(2, dim=-1)
        post_sigma = F.softplus(post_log_sigma) + 0.1
        
        # Reparameterization trick で z をサンプリング
        z = post_mu + post_sigma * torch.randn_like(post_sigma)
        
        return h, z, prior_mu, prior_sigma, post_mu, post_sigma
    
    def imagine_step(self, h_prev, z_prev, action):
        # 1ステップの状態更新（観測なし = 事前分布のみ）
        gru_input = torch.cat([z_prev, action], dim=-1)
        h = self.gru(gru_input, h_prev)
        
        prior_params = self.prior_net(h)
        prior_mu, prior_log_sigma = prior_params.chunk(2, dim=-1)
        prior_sigma = F.softplus(prior_log_sigma) + 0.1
        
        z = prior_mu + prior_sigma * torch.randn_like(prior_sigma)
        return h, z

# テスト
rssm = RSSM()
h, z = rssm.initial_state(batch_size=4)
action = torch.randn(4, 1)
obs = torch.randn(4, 5)

h_new, z_new, pm, ps, qm, qs = rssm.observe_step(h, z, action, obs)
print(f"確定的状態 h: {h_new.shape}")
print(f"確率的状態 z: {z_new.shape}")
print(f"事前分布: mu={pm.shape}, sigma={ps.shape}")
print(f"事後分布: mu={qm.shape}, sigma={qs.shape}")
print(f"パラメータ数: {sum(p.numel() for p in rssm.parameters()):,}")

<a id="section4"></a>
## 4. 世界モデルの構築と訓練

世界モデルは以下の4つの要素から構成されます：

| コンポーネント | 役割 | 入力 → 出力 |
|---------------|------|------------|
| **エンコーダ** | 観測を埋め込み | o_t → embed_t |
| **RSSM** | 状態遷移 | (h, z, a) → (h', z') |
| **デコーダ** | 観測の再構成 | (h, z) → ô_t |
| **報酬予測** | 報酬の予測 | (h, z) → r̂_t |

### 損失関数

$$\mathcal{L} = \underbrace{\|o_t - \hat{o}_t\|^2}_{\text{再構成損失}} + \underbrace{\|r_t - \hat{r}_t\|^2}_{\text{報酬損失}} + \beta \underbrace{D_{KL}(q \| p)}_{\text{KLダイバージェンス}}$$

In [ ]:
# ============================================================
# 世界モデル（World Model）の実装
# ============================================================

class WorldModel(nn.Module):
    # エンコーダ + RSSM + デコーダ + 報酬予測
    
    def __init__(self, obs_dim=5, action_dim=1, det_dim=32, stoch_dim=16):
        super().__init__()
        self.obs_dim = obs_dim
        self.rssm = RSSM(obs_dim, action_dim, det_dim, stoch_dim)
        
        # デコーダ: (h, z) → 観測の再構成
        state_dim = det_dim + stoch_dim
        self.decoder = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ELU(),
            nn.Linear(64, 64), nn.ELU(),
            nn.Linear(64, obs_dim)
        )
        
        # 報酬予測: (h, z) → 報酬
        self.reward_predictor = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ELU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, observations, actions):
        # シーケンス全体を処理
        # observations: (B, T, obs_dim)
        # actions: (B, T, action_dim)
        B, T, _ = observations.shape
        
        h, z = self.rssm.initial_state(B)
        
        all_h, all_z = [], []
        all_prior_mu, all_prior_sigma = [], []
        all_post_mu, all_post_sigma = [], []
        
        for t in range(T):
            obs_t = observations[:, t]
            act_t = actions[:, t]
            h, z, pm, ps, qm, qs = self.rssm.observe_step(h, z, act_t, obs_t)
            
            all_h.append(h)
            all_z.append(z)
            all_prior_mu.append(pm)
            all_prior_sigma.append(ps)
            all_post_mu.append(qm)
            all_post_sigma.append(qs)
        
        # スタック: (B, T, dim)
        H = torch.stack(all_h, dim=1)
        Z = torch.stack(all_z, dim=1)
        state = torch.cat([H, Z], dim=-1)
        
        # デコード
        obs_pred = self.decoder(state)
        reward_pred = self.reward_predictor(state).squeeze(-1)
        
        # KLダイバージェンス
        prior_mu = torch.stack(all_prior_mu, dim=1)
        prior_sigma = torch.stack(all_prior_sigma, dim=1)
        post_mu = torch.stack(all_post_mu, dim=1)
        post_sigma = torch.stack(all_post_sigma, dim=1)
        
        kl = torch.log(prior_sigma / post_sigma) +              (post_sigma**2 + (post_mu - prior_mu)**2) / (2 * prior_sigma**2) - 0.5
        kl = kl.sum(dim=-1).mean()
        
        return obs_pred, reward_pred, kl, H, Z

# テスト
world_model = WorldModel()
obs_seq = torch.randn(4, 20, 5)
act_seq = torch.randn(4, 20, 1)
obs_pred, rew_pred, kl, H, Z = world_model(obs_seq, act_seq)
print(f"観測予測: {obs_pred.shape}")
print(f"報酬予測: {rew_pred.shape}")
print(f"KL: {kl.item():.4f}")
print(f"パラメータ数: {sum(p.numel() for p in world_model.parameters()):,}")

In [ ]:
# ============================================================
# 世界モデルの訓練
# ============================================================

def prepare_batch(episodes, batch_size=32, seq_len=20):
    # エピソードからバッチを生成
    batch_obs, batch_act, batch_rew = [], [], []
    for _ in range(batch_size):
        ep = episodes[np.random.randint(len(episodes))]
        T = len(ep['observations'])
        if T <= seq_len:
            start = 0
        else:
            start = np.random.randint(0, T - seq_len)
        batch_obs.append(ep['observations'][start:start+seq_len])
        batch_act.append(ep['actions'][start:start+seq_len].reshape(-1, 1))
        batch_rew.append(ep['rewards'][start:start+seq_len])
    return (torch.tensor(np.array(batch_obs), dtype=torch.float32),
            torch.tensor(np.array(batch_act), dtype=torch.float32),
            torch.tensor(np.array(batch_rew), dtype=torch.float32))

# 訓練ループ
world_model = WorldModel()
optimizer = torch.optim.Adam(world_model.parameters(), lr=3e-4)
kl_beta = 0.1

losses = {'total': [], 'recon': [], 'reward': [], 'kl': []}

print("世界モデルの訓練開始...")
for epoch in range(100):
    obs_b, act_b, rew_b = prepare_batch(data, batch_size=32, seq_len=20)
    
    obs_pred, rew_pred, kl, _, _ = world_model(obs_b, act_b)
    
    recon_loss = F.mse_loss(obs_pred, obs_b)
    reward_loss = F.mse_loss(rew_pred, rew_b)
    total_loss = recon_loss + reward_loss + kl_beta * kl
    
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(world_model.parameters(), 100.0)
    optimizer.step()
    
    losses['total'].append(total_loss.item())
    losses['recon'].append(recon_loss.item())
    losses['reward'].append(reward_loss.item())
    losses['kl'].append(kl.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1}: total={total_loss.item():.4f}, "
              f"recon={recon_loss.item():.4f}, reward={reward_loss.item():.4f}, "
              f"kl={kl.item():.4f}")

print("✅ 世界モデルの訓練完了")

In [ ]:
# ============================================================
# 訓練損失の可視化
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, vals) in zip(axes.flatten(), losses.items()):
    ax.plot(vals, linewidth=2, alpha=0.7)
    ax.set_title(f'{name} 損失', fontsize=13, fontweight='bold')
    ax.set_xlabel('エポック')
    ax.set_ylabel('損失')
    ax.grid(True, alpha=0.3)

plt.suptitle('世界モデルの訓練損失', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("観察: 再構成損失とKL損失が低下し、世界モデルが環境のダイナミクスを学習している")

<a id="section5"></a>
## 5. Actor-Critic in 潜在空間

DreamerV3では、Actor（方策）とCritic（価値関数）を**潜在空間内**で訓練します。
実環境とのインタラクションは不要で、すべてRSSM内の「想像」で行います。

### Actor（方策ネットワーク）
$$\pi(a_t | s_t) = \mathcal{N}(\mu_\pi(s_t), \sigma_\pi(s_t))$$

### Critic（価値ネットワーク）
$$V(s_t) \approx \mathbb{E}\left[\sum_{k=0}^{H} \gamma^k r_{t+k}\right]$$

In [ ]:
# ============================================================
# Actor と Critic の実装
# ============================================================

class Actor(nn.Module):
    # 方策ネットワーク: 潜在状態 → 行動分布
    
    def __init__(self, state_dim=48, action_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ELU(),
            nn.Linear(64, 64), nn.ELU(),
        )
        self.mu_head = nn.Linear(64, action_dim)
        self.log_sigma_head = nn.Linear(64, action_dim)
    
    def forward(self, state):
        features = self.net(state)
        mu = torch.tanh(self.mu_head(features))  # [-1, 1]
        log_sigma = self.log_sigma_head(features)
        sigma = F.softplus(log_sigma) + 0.01
        return Normal(mu, sigma)

class Critic(nn.Module):
    # 価値ネットワーク: 潜在状態 → 状態価値
    
    def __init__(self, state_dim=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ELU(),
            nn.Linear(64, 64), nn.ELU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, state):
        return self.net(state).squeeze(-1)

# テスト
state_dim = 32 + 16  # det_dim + stoch_dim
actor = Actor(state_dim)
critic = Critic(state_dim)
dummy_state = torch.randn(4, state_dim)

action_dist = actor(dummy_state)
print(f"Actor出力: mu={action_dist.loc.shape}, sigma={action_dist.scale.shape}")
print(f"サンプル行動: {action_dist.sample()}")

value = critic(dummy_state)
print(f"Critic出力: {value.shape}")
print(f"パラメータ数: Actor={sum(p.numel() for p in actor.parameters()):,}, "
      f"Critic={sum(p.numel() for p in critic.parameters()):,}")

<a id="section6"></a>
## 6. Imagination Rollout

Imagination Rolloutは、RSSMの中で**行動→遷移→報酬**のシーケンスを展開する処理です。
実環境は一切使わず、世界モデルだけで未来の軌跡を「想像」します。

```
想像ロールアウト（H=15ステップ）:
  s_0 → a_0 → s_1 → a_1 → ... → s_H
       ↑          ↑
     Actor        RSSM
       ↓          ↓
     r̂_0         r̂_1 → ... → r̂_H
```

In [ ]:
# ============================================================
# Imagination Rollout の実装
# ============================================================

def imagine_rollout(rssm, actor, reward_predictor, h_init, z_init, 
                    horizon=15, det_dim=32, stoch_dim=16):
    # 世界モデル内で軌跡を想像する
    # 実環境不要 - すべて潜在空間内で完結
    
    states = []      # (h, z) の連結
    rewards = []     # 予測報酬
    log_probs = []   # 方策のlog確率
    
    h, z = h_init, z_init
    
    for t in range(horizon):
        state = torch.cat([h, z], dim=-1)
        states.append(state)
        
        # Actorから行動をサンプリング
        action_dist = actor(state)
        action = action_dist.rsample()  # reparameterized sample
        log_prob = action_dist.log_prob(action).sum(dim=-1)
        log_probs.append(log_prob)
        
        # 報酬予測
        reward = reward_predictor(state).squeeze(-1)
        rewards.append(reward)
        
        # RSSMで次の状態を想像（観測なし = 事前分布）
        h, z = rssm.imagine_step(h, z, action)
    
    states = torch.stack(states, dim=1)    # (B, H, state_dim)
    rewards = torch.stack(rewards, dim=1)  # (B, H)
    log_probs = torch.stack(log_probs, dim=1)
    
    return states, rewards, log_probs

# テスト
world_model.eval()
h_test, z_test = world_model.rssm.initial_state(8)
with torch.no_grad():
    states, rewards, log_probs = imagine_rollout(
        world_model.rssm, actor, world_model.reward_predictor,
        h_test, z_test, horizon=15
    )
print(f"想像された状態系列: {states.shape}")
print(f"想像された報酬系列: {rewards.shape}")
print(f"平均想像報酬: {rewards.mean().item():.4f}")

<a id="section7"></a>
## 7. SimpleDreamer 統合パイプライン

3つのフェーズを統合して完全なDreamerエージェントを構築します。

In [ ]:
# ============================================================
# SimpleDreamer 統合パイプライン
# ============================================================

class SimpleDreamer:
    # DreamerV3の簡易実装
    
    def __init__(self, obs_dim=5, action_dim=1, det_dim=32, stoch_dim=16):
        self.state_dim = det_dim + stoch_dim
        
        # 世界モデル
        self.world_model = WorldModel(obs_dim, action_dim, det_dim, stoch_dim)
        
        # Actor-Critic
        self.actor = Actor(self.state_dim, action_dim)
        self.critic = Critic(self.state_dim)
        
        # オプティマイザ
        self.wm_optim = torch.optim.Adam(self.world_model.parameters(), lr=3e-4)
        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=1e-4)
        self.critic_optim = torch.optim.Adam(self.critic.parameters(), lr=1e-4)
        
        self.gamma = 0.99
        self.lambda_ = 0.95
        self.horizon = 15
        self.det_dim = det_dim
        self.stoch_dim = stoch_dim
    
    def train_world_model(self, episodes, n_epochs=50, batch_size=32):
        # フェーズ1: 世界モデル訓練
        losses = []
        for epoch in range(n_epochs):
            obs_b, act_b, rew_b = prepare_batch(episodes, batch_size)
            obs_pred, rew_pred, kl, _, _ = self.world_model(obs_b, act_b)
            
            loss = F.mse_loss(obs_pred, obs_b) + F.mse_loss(rew_pred, rew_b) + 0.1 * kl
            self.wm_optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.world_model.parameters(), 100.0)
            self.wm_optim.step()
            losses.append(loss.item())
        return losses
    
    def train_actor_critic(self, episodes, n_iter=50, batch_size=16):
        # フェーズ2: 想像の中でActor-Criticを訓練
        actor_losses, critic_losses = [], []
        
        for it in range(n_iter):
            # 実データから初期状態を取得
            obs_b, act_b, _ = prepare_batch(episodes, batch_size, seq_len=5)
            with torch.no_grad():
                _, _, _, H, Z = self.world_model(obs_b, act_b)
                h_init = H[:, -1]  # 最後のタイムステップ
                z_init = Z[:, -1]
            
            # 想像ロールアウト
            states, rewards, log_probs = imagine_rollout(
                self.world_model.rssm, self.actor,
                self.world_model.reward_predictor,
                h_init, z_init, self.horizon
            )
            
            # Lambda Returns の計算
            values = self.critic(states).detach()
            returns = torch.zeros_like(rewards)
            last_return = values[:, -1]
            for t in reversed(range(self.horizon)):
                returns[:, t] = rewards[:, t] + self.gamma * (
                    self.lambda_ * last_return + (1 - self.lambda_) * values[:, t])
                last_return = returns[:, t]
            
            # Critic更新
            values_pred = self.critic(states)
            critic_loss = F.mse_loss(values_pred, returns.detach())
            self.critic_optim.zero_grad()
            critic_loss.backward()
            self.critic_optim.step()
            
            # Actor更新（方策勾配）
            actor_loss = -(returns.detach() * log_probs).mean()
            self.actor_optim.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 100.0)
            self.actor_optim.step()
            
            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())
        
        return actor_losses, critic_losses

# Dreamerの作成と訓練
dreamer = SimpleDreamer()

print("=== フェーズ1: 世界モデル訓練 ===")
wm_losses = dreamer.train_world_model(data, n_epochs=80)
print(f"  最終損失: {wm_losses[-1]:.4f}")

print("\n=== フェーズ2: Actor-Critic訓練 (想像の中) ===")
a_losses, c_losses = dreamer.train_actor_critic(data, n_iter=80)
print(f"  Actor最終損失: {a_losses[-1]:.4f}")
print(f"  Critic最終損失: {c_losses[-1]:.4f}")

print("\n✅ SimpleDreamer訓練完了")

In [ ]:
# ============================================================
# 学習した方策の評価
# ============================================================

def evaluate_policy(dreamer, env, n_episodes=20):
    # 学習した方策で環境を評価
    rewards_list = []
    trajectories = []
    
    for ep in range(n_episodes):
        obs = env.reset()
        h, z = dreamer.world_model.rssm.initial_state(1)
        total_reward = 0
        positions = [obs[0]]
        
        for step in range(50):
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            act_dummy = torch.zeros(1, 1)
            
            with torch.no_grad():
                h, z, _, _, _, _ = dreamer.world_model.rssm.observe_step(
                    h, z, act_dummy, obs_t)
                state = torch.cat([h, z], dim=-1)
                action_dist = dreamer.actor(state)
                action = action_dist.loc  # 平均を使用（評価時）
            
            obs, reward, done = env.step(action.numpy().flatten()[0])
            total_reward += reward
            positions.append(obs[0])
            if done:
                break
        
        rewards_list.append(total_reward)
        trajectories.append(positions)
    
    return rewards_list, trajectories

# ランダム方策との比較
env = BallBalanceEnv()
dreamer_rewards, dreamer_traj = evaluate_policy(dreamer, env)
random_rewards = [np.sum(ep['rewards']) for ep in data[:20]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 報酬の比較
axes[0].boxplot([random_rewards, dreamer_rewards], labels=['ランダム', 'Dreamer'])
axes[0].set_ylabel('累積報酬', fontsize=12)
axes[0].set_title('方策の性能比較', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 軌跡の可視化
for traj in dreamer_traj[:5]:
    axes[1].plot(traj, alpha=0.5, linewidth=2)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1, label='目標位置')
axes[1].set_xlabel('ステップ', fontsize=12)
axes[1].set_ylabel('ボール位置', fontsize=12)
axes[1].set_title('Dreamer方策による軌跡', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ランダム方策: 平均報酬 = {np.mean(random_rewards):.2f}")
print(f"Dreamer方策:  平均報酬 = {np.mean(dreamer_rewards):.2f}")

<a id="section8"></a>
## 8. 夢の可視化

DreamerV3が「夢の中」で想像する軌跡を可視化します。
実環境の軌跡と比較することで、世界モデルの精度を確認できます。

In [ ]:
# ============================================================
# 「夢」の可視化 — 想像された軌跡 vs 実軌跡
# ============================================================

def visualize_dreams(dreamer, data, n_dreams=5):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for i in range(min(n_dreams, 6)):
        ax = axes.flatten()[i]
        ep = data[i]
        T = len(ep['observations'])
        
        # 実軌跡
        real_obs = ep['observations'][:, 0]  # 位置のみ
        
        # 世界モデルで想像
        obs_t = torch.tensor(ep['observations'][:20], dtype=torch.float32).unsqueeze(0)
        act_t = torch.tensor(ep['actions'][:20].reshape(-1, 1), dtype=torch.float32).unsqueeze(0)
        
        with torch.no_grad():
            obs_pred, _, _, H, Z = dreamer.world_model(obs_t, act_t)
            imagined_obs = obs_pred[0, :, 0].numpy()
        
        ax.plot(range(min(T, 20)), real_obs[:20], 'b-o', markersize=3, 
               linewidth=2, label='実軌跡', alpha=0.8)
        ax.plot(range(20), imagined_obs, 'r--s', markersize=3,
               linewidth=2, label='想像(夢)', alpha=0.8)
        ax.axhline(y=0, color='green', linestyle=':', alpha=0.5)
        ax.set_title(f'エピソード {i+1}', fontsize=12, fontweight='bold')
        ax.set_xlabel('ステップ')
        ax.set_ylabel('ボール位置')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('夢の中の軌跡 vs 実際の軌跡', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("青線: 実際に環境で起きた軌跡")
    print("赤線: 世界モデルが想像した軌跡")
    print("→ 世界モデルが環境のダイナミクスをある程度正確に学習している")

visualize_dreams(dreamer, data)

In [ ]:
# ============================================================
# 訓練過程の総合可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 世界モデル損失
axes[0].plot(wm_losses, linewidth=2, color='steelblue')
axes[0].set_title('世界モデル損失', fontsize=13, fontweight='bold')
axes[0].set_xlabel('エポック')
axes[0].set_ylabel('損失')
axes[0].grid(True, alpha=0.3)

# Actor損失
axes[1].plot(a_losses, linewidth=2, color='coral')
axes[1].set_title('Actor損失', fontsize=13, fontweight='bold')
axes[1].set_xlabel('イテレーション')
axes[1].set_ylabel('損失')
axes[1].grid(True, alpha=0.3)

# Critic損失
axes[2].plot(c_losses, linewidth=2, color='mediumseagreen')
axes[2].set_title('Critic損失', fontsize=13, fontweight='bold')
axes[2].set_xlabel('イテレーション')
axes[2].set_ylabel('損失')
axes[2].grid(True, alpha=0.3)

plt.suptitle('SimpleDreamer 訓練過程', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

<a id="summary"></a>
## 9. まとめ・よくあるエラー・自己評価クイズ

### 🎯 このノートブックで学んだこと

| 概念 | 内容 |
|------|------|
| **RSSM** | 確定的GRU + 確率的ガウシアンの組み合わせで状態遷移をモデル化 |
| **世界モデル** | エンコーダ + RSSM + デコーダ + 報酬予測の4要素 |
| **3フェーズ学習** | 世界モデル訓練 → 想像の中で方策学習 → データ収集 |
| **Imagination Rollout** | 実環境不要で潜在空間内の未来軌跡を生成 |
| **Actor-Critic** | 潜在空間内で方策と価値関数を訓練 |

### ⚠️ よくあるエラー

#### エラー#1: KLダイバージェンスの爆発

```python
# ❌ betaが大きすぎるとKLが支配的になり、再構成品質が低下
loss = recon_loss + 10.0 * kl  # beta=10は大きすぎ

# ✅ 小さめのbetaから始める
loss = recon_loss + 0.1 * kl   # beta=0.1
```

#### エラー#2: 想像のホライズンが長すぎる

```python
# ❌ 長すぎると誤差が蓄積
states, rewards, _ = imagine_rollout(rssm, actor, reward_pred, h, z, horizon=100)

# ✅ 15-30ステップ程度が適切
states, rewards, _ = imagine_rollout(rssm, actor, reward_pred, h, z, horizon=15)
```

#### エラー#3: 勾配クリッピングの不足

```python
# ❌ クリッピングなしでは勾配爆発の危険
loss.backward()
optimizer.step()

# ✅ 必ず勾配クリッピングを行う
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
optimizer.step()
```

## 🎓 自己評価クイズ

### Q1: RSSMの確定的状態(h)と確率的状態(z)はそれぞれどのような役割を持ちますか？

<details>
<summary>💡 答えを見る</summary>

**確定的状態 h（GRU隠れ状態）**: 長期的な時系列依存関係を保持する決定論的な記憶。
**確率的状態 z（ガウシアン潜在変数）**: 環境の不確実性や多様性を表現する確率的な要素。

両者を組み合わせることで、決定論的に予測できる部分と確率的な変動の両方をモデル化できます。

</details>

---

### Q2: DreamerV3の3つのフェーズを順番に説明してください。

<details>
<summary>💡 答えを見る</summary>

1. **世界モデル学習**: 実データ(観測, 行動, 報酬)を使ってRSSM+デコーダ+報酬予測を訓練
2. **想像の中で方策学習**: 学習した世界モデル内でロールアウトし、Actor-Criticを勾配ベースで更新
3. **データ収集**: 学習した方策で実環境と対話し、新しいデータを収集

</details>

---

### Q3: 事前分布(Prior)と事後分布(Posterior)の違いは？なぜ両方必要ですか？

<details>
<summary>💡 答えを見る</summary>

**事前分布**: 観測なしで、確定的状態hのみから次の状態zを予測
**事後分布**: 観測を使って、より正確な状態推定を行う

訓練時は事後分布を使い、想像時は事前分布を使います。KLダイバージェンスで両者を近づけることで、
想像時（観測なし）でも精度の高い予測が可能になります。

</details>

---

### Q4: Imagination Rolloutの利点は何ですか？

<details>
<summary>💡 答えを見る</summary>

- **サンプル効率**: 実環境とのインタラクションなしで大量の「経験」を生成できる
- **並列化**: バッチ処理で同時に多数の想像軌跡を生成可能
- **安全性**: 実環境でリスクのある行動を試す前に、想像の中で結果を予測できる
- **勾配ベース学習**: 微分可能なので、方策勾配法で直接Actor-Criticを更新できる

</details>

---

### Q5: Dyna-Q（Notebook 142）とDreamerV3の最大の違いは何ですか？

<details>
<summary>💡 答えを見る</summary>

最大の違いは**状態表現と学習方法**です：

- **Dyna-Q**: テーブルベース、1ステップの想像、Q値の離散更新
- **DreamerV3**: ニューラルネット潜在空間、多ステップロールアウト、勾配ベースのActor-Critic更新

DreamerV3は連続状態・連続行動空間を扱え、画像観測にも対応できるため、
現実世界の複雑なタスクに適用可能です。

</details>

---

### ✅ 学習チェックリスト

- [ ] RSSMの確定的/確率的パスの役割を説明できる
- [ ] 世界モデルの4コンポーネント（エンコーダ/RSSM/デコーダ/報酬予測）を実装できる
- [ ] KLダイバージェンスの役割を説明できる
- [ ] Imagination Rolloutをスクラッチで実装できた
- [ ] Actor-Criticを潜在空間で訓練する流れを理解した
- [ ] Dyna-QとDreamerV3の違いを3つ以上挙げられる

---

**次のステップ**: Notebook 144では、**Genie** — 動画から行動ラベルなしで操作可能な世界を学ぶ手法を学びます！